In [ ]:
from google.colab import drive
import os


drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/Autism_Agent_Project'

video_training_path = os.path.join(base_path, 'data_video_skeleton/training_set')
audio_path = os.path.join(base_path, 'data_audio_noise/UrbanSound8K/audio')

# (Sanity Check)
print("\n--- HASIL PENGECEKAN DATA ---")

if os.path.exists(video_training_path):
    print("✅ Folder Video Training DITEMUKAN!")
    # Tampilkan 5 isi pertama foldernya untuk memastikan ada ASD dan TD
    print("   Isi folder:", os.listdir(video_training_path)[:5])
else:
    print("❌ Folder Video TIDAK ditemukan.")

# Cek Folder Audio
if os.path.exists(audio_path):
    print("✅ Folder Audio Noise DITEMUKAN!")
    print("   Ada", len(os.listdir(audio_path)), "folder/file di dalamnya.")
else:
    print("❌ Folder Audio TIDAK ditemukan. (Pastikan ada folder 'UrbanSound8K' di dalam 'data_audio_noise')")

Mounted at /content/drive

--- HASIL PENGECEKAN DATA ---
✅ Folder Video Training DITEMUKAN!
   Isi folder: ['ASD', 'TD']
✅ Folder Audio Noise DITEMUKAN!
   Ada 6 folder/file di dalamnya.


In [2]:
import os

# 1. Define the workspace directory in Colab's local storage
# Note: We clone to '/content/' (local VM storage) because it's much faster
# than running code directly from Google Drive.
workspace_dir = '/content/Autism_Agent_Workspace'

if not os.path.exists(workspace_dir):
    os.makedirs(workspace_dir)

# Change current working directory to the workspace
os.chdir(workspace_dir)

# 2. Clone the GitHub Repositories
print("--- DOWNLOADING MACHINE LEARNING MODELS ---")
print("Cloning Video Model Repository (VGG16 + LSTM)...")
!git clone https://github.com/AutismBrainBehavior/Video-Neural-Network-ASD-screening.git

print("\nCloning Audio Model Repository (CNN)...")
!git clone https://github.com/AutismBrainBehavior/Audio-Neural-Network-ASD-screening.git

# 3. Verify the cloned directories and list their contents
print("\n--- WORKSPACE CONTENT ---")
!ls -la

print("\n--- INSIDE VIDEO REPOSITORY ---")
!ls -la Video-Neural-Network-ASD-screening/

--- DOWNLOADING MACHINE LEARNING MODELS ---
Cloning Video Model Repository (VGG16 + LSTM)...
Cloning into 'Video-Neural-Network-ASD-screening'...
remote: Enumerating objects: 137, done.
remote: Counting objects: 100% (137/137), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 137 (delta 63), reused 112 (delta 46), pack-reused 0 (from 0)
Receiving objects: 100% (137/137), 1.21 MiB | 4.12 MiB/s, done.
Resolving deltas: 100% (63/63), done.

Cloning Audio Model Repository (CNN)...
Cloning into 'Audio-Neural-Network-ASD-screening'...
remote: Enumerating objects: 40, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 40 (delta 14), reused 23 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (40/40), 367.81 KiB | 2.01 MiB/s, done.
Resolving deltas: 100% (14/14), done.

--- WORKSPACE CONTENT ---
total 16
drwxr-xr-x 4 root root 4096 Mar  6 11:58 .
drwxr-xr-x 1 root root 4096 Mar  6 11:58 ..
drwxr-xr-x 7 

In [3]:
# 1. Search inside the Video repository for files defining the Neural Network
# We are looking for keywords like 'VGG16', 'LSTM', or 'Dense' layer definitions.
print("--- SEARCHING FOR MODEL ARCHITECTURE IN VIDEO REPO ---")
!grep -rnw 'Video-Neural-Network-ASD-screening/' -e 'VGG16' -e 'LSTM' -e 'Dense' --include=\*.py

print("\n--- SEARCHING FOR MODEL ARCHITECTURE IN AUDIO REPO ---")
!grep -rnw 'Audio-Neural-Network-ASD-screening/' -e 'Conv1D' -e 'Dense' --include=\*.py

--- SEARCHING FOR MODEL ARCHITECTURE IN VIDEO REPO ---
Video-Neural-Network-ASD-screening/convolutional.py:4:from keras.layers import Conv2D, Activation, MaxPooling2D, Dropout, Flatten, Dense
Video-Neural-Network-ASD-screening/convolutional.py:63:        model.add(Dense(units=512))
Video-Neural-Network-ASD-screening/convolutional.py:66:        model.add(Dense(units=nb_classes))
Video-Neural-Network-ASD-screening/recurrent_networks.py:1:from keras.layers import Dense, Activation, Dropout, Bidirectional
Video-Neural-Network-ASD-screening/recurrent_networks.py:2:from keras.layers.recurrent import LSTM
Video-Neural-Network-ASD-screening/recurrent_networks.py:4:from keras.applications.vgg16 import VGG16
Video-Neural-Network-ASD-screening/recurrent_networks.py:57:        model.add(Bidirectional(LSTM(units=HIDDEN_UNITS, return_sequences=True),
Video-Neural-Network-ASD-screening/recurrent_networks.py:59:        model.add(Bidirectional(LSTM(10)))
Video-Neural-Network-ASD-screening/recurrent_net

In [ ]:
from keras.models import Model
from keras.layers import LSTM, Dense, Input

print("--- 1. BUILDING THE ORIGINAL MODEL (FUNCTIONAL API) ---")

# 1. Input Layer: Accepts variable-length sequences of 50-dimensional skeleton coordinates
video_input = Input(shape=(None, 50), name='video_input_gate')

# 2. LSTM Layer: Processes the temporal sequence of skeleton movements
lstm_layer = LSTM(units=128, return_sequences=False, name='skeleton_tracker_lstm')(video_input)

# 3. Dense Layer (Feature Extractor): Compiles the temporal data into a 512-dimensional feature vector
behavior_features = Dense(512, activation='relu', name='behavior_features_512')(lstm_layer)

# 4. Output Layer (Classifier): Outputs the probability of ASD vs TD (This will be removed later)
asd_classifier = Dense(2, activation='softmax', name='asd_classifier_mouth')(behavior_features)

# Construct the full original model
original_video_model = Model(inputs=video_input, outputs=asd_classifier, name="Original_Video_Classifier")

print("\nOriginal Video Model Architecture:")
original_video_model.summary()


print("\n\n--- 2. CREATING THE HEADLESS FEATURE EXTRACTOR ---")
# By using the Functional API, we can easily create a new model that shares the same input
# but terminates early at the 'behavior_features_512' layer, bypassing the final classifier.
headless_video_model = Model(inputs=video_input, outputs=behavior_features, name="Headless_Feature_Extractor")

print("\nHeadless Video Model Architecture (Classifier removed):")
headless_video_model.summary()

--- 1. BUILDING THE ORIGINAL MODEL (FUNCTIONAL API) ---

Original Video Model Architecture:


Model: "Original_Video_Classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ video_input_gate (InputLayer)   │ (None, None, 50)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ skeleton_tracker_lstm (LSTM)    │ (None, 128)            │        91,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ behavior_features_512 (Dense)   │ (None, 512)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ asd_classifier_mouth (Dense)    │ (None, 2)              │         1,026 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 158,722 (620.01 KB)

 Trainable params: 158,722 (620.01 KB)

...

Model: "Headless_Feature_Extractor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ video_input_gate (InputLayer)   │ (None, None, 50)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ skeleton_tracker_lstm (LSTM)    │ (None, 128)            │        91,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ behavior_features_512 (Dense)   │ (None, 512)            │        66,048 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 157,696 (616.00 KB)

 Trainable params: 157,696 (616.00 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
import os
import numpy as np

print("--- 1. LOCATING THE SKELETON DATA ---")
# Define the path to the ASD training folder
asd_folder_path = '/content/drive/MyDrive/Autism_Agent_Project/data_video_skeleton/training_set/ASD'

# List the files inside the folder
files_in_asd = os.listdir(asd_folder_path)
print(f"Total files found in ASD folder: {len(files_in_asd)}")

if len(files_in_asd) > 0:
    print(f"First 5 files: {files_in_asd[:5]}")

    print("\n--- 2. INSPECTING THE FIRST FILE ---")
    first_file = files_in_asd[0]
    first_file_path = os.path.join(asd_folder_path, first_file)
    print(f"Target File: {first_file}")

#     # Check if the file is a Numpy array (.npy), which is standard for skeleton data
#     if first_file.endswith('.npy'):
#         try:
#             sample_data = np.load(first_file_path)
#             print("✅ Successfully loaded as Numpy Array!")
#             print(f"📊 DATA SHAPE (Bentuk Data): {sample_data.shape}")
#             print(f"Data type: {sample_data.dtype}")
#         except Exception as e:
#             print(f"❌ Failed to load .npy file. Error: {e}")
#     else:
#         print(f"⚠️ The file is NOT a .npy file. It has a different extension.")
# else:
#     print("❌ ASD folder is empty or path is incorrect.")

--- 1. LOCATING THE SKELETON DATA ---
Total files found in ASD folder: 4840
First 5 files: ['Subj_32_part_101.mp4', 'Subj_32_part_104.mp4', 'Subj_32_part_76.mp4', 'Subj_32_part_96.mp4', 'Subj_32_part_103.mp4']

--- 2. INSPECTING THE FIRST FILE ---
Target File: Subj_32_part_101.mp4


In [6]:
import os
import cv2
import numpy as np

print("--- 1. LOCATING THE MP4 FILE ---")
asd_folder_path = '/content/drive/MyDrive/Autism_Agent_Project/data_video_skeleton/training_set/ASD'
# Menggunakan nama file pertama dari hasil screenshot Anda
target_file = 'Subj_32_part_101.mp4'
video_path = os.path.join(asd_folder_path, target_file)

print(f"Target Video: {video_path}")

print("\n--- 2. EXTRACTING FRAMES (USING OPENCV) ---")
# Open the video file
cap = cv2.VideoCapture(video_path)
frames = []

# Loop to read each frame of the video
while True:
    ret, frame = cap.read()
    if not ret:
        break # Exit loop if video ends

    # Resize frame to 224x224 pixels (Standard requirement for VGG16 model)
    frame_resized = cv2.resize(frame, (224, 224))

    # Keras/VGG16 expects RGB format, OpenCV uses BGR by default, so we convert it
    frame_rgb = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)
    frames.append(frame_rgb)

# Close the video file
cap.release()

# Convert the list of frames into a mathematical Numpy array
video_array = np.array(frames)

print("✅ Video Successfully Extracted!")
print(f"📊 RAW VIDEO TENSOR SHAPE: {video_array.shape}")
print("Format: (Number of Frames, Image Width, Image Height, RGB Channels)")

--- 1. LOCATING THE MP4 FILE ---
Target Video: /content/drive/MyDrive/Autism_Agent_Project/data_video_skeleton/training_set/ASD/Subj_32_part_101.mp4

--- 2. EXTRACTING FRAMES (USING OPENCV) ---
✅ Video Successfully Extracted!
📊 RAW VIDEO TENSOR SHAPE: (144, 224, 224, 3)
Format: (Number of Frames, Image Width, Image Height, RGB Channels)


In [7]:
from keras.applications.vgg16 import VGG16, preprocess_input
import numpy as np

print("--- 1. BUILDING THE 'EYES' (VGG16 FEATURE EXTRACTOR) ---")
# We load the pre-trained VGG16 model.
# include_top=False means we cut off ITS mouth too (we only want the eyes).
# pooling='avg' flattens the image into a neat 1D array of 512 numbers.
vgg16_eyes = VGG16(weights='imagenet', include_top=False, pooling='avg')

print("\n--- 2. TRANSLATING FRAMES INTO MATH (FEATURE EXTRACTION) ---")
print("Preparing 144 frames for the VGG16 eyes...")

# VGG16 requires a specific color format, so we preprocess the video_array we made earlier
preprocessed_frames = preprocess_input(video_array)

# The Eyes look at the 144 frames and extract the features!
# Note: This might take a few seconds depending on the Colab GPU.
extracted_features = vgg16_eyes.predict(preprocessed_frames)

print("\n✅ Features Successfully Extracted by VGG16!")
print(f"📊 NEW FEATURE SHAPE: {extracted_features.shape}")
print("Format: (Number of Frames, Features per Frame)")

--- 1. BUILDING THE 'EYES' (VGG16 FEATURE EXTRACTOR) ---
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

--- 2. TRANSLATING FRAMES INTO MATH (FEATURE EXTRACTION) ---
Preparing 144 frames for the VGG16 eyes...
5/5 ━━━━━━━━━━━━━━━━━━━━ 19s 2s/step

✅ Features Successfully Extracted by VGG16!
📊 NEW FEATURE SHAPE: (144, 512)
Format: (Number of Frames, Features per Frame)


In [8]:
from keras.models import Model
from keras.layers import LSTM, Dense, Input
import numpy as np

print("--- 1. UPGRADING THE LSTM ROBOT (ADJUSTING THE PLUG) ---")
# Adjusting the input to match the 512 features from VGG16
video_feature_input = Input(shape=(None, 512), name='vgg16_feature_input')

# The Eyes & Memory (Now receiving the 512 visual notes per frame)
lstm_layer = LSTM(units=128, return_sequences=False, name='skeleton_tracker_lstm')(video_feature_input)

# The Brain (Summarizing the entire video sequence into 1 vector)
behavior_features = Dense(512, activation='relu', name='final_behavior_features')(lstm_layer)

# The New Upgraded Headless Model
final_headless_model = Model(inputs=video_feature_input, outputs=behavior_features, name="Final_Video_Feature_Extractor")

print("✅ LSTM Robot Upgraded successfully!\n")

print("--- 2. FEEDING DATA TO THE LSTM BRAIN ---")
# AI models always expect a "batch" (batch_size, timesteps, features)
# So we wrap our (144, 512) data into (1 video, 144 frames, 512 features)
final_input_data = np.expand_dims(extracted_features, axis=0)

print(f"Feeding data with shape: {final_input_data.shape} into the LSTM...")

# THE FINAL EXTRACTION! (The robot watches the sequence and concludes)
agent_ready_features = final_headless_model.predict(final_input_data)

print("\n🎉 BINGO! FINAL FEATURES EXTRACTED! 🎉")
print(f"📊 AGENT-READY SHAPE: {agent_ready_features.shape}")
print("Format: (Number of Videos, Extracted Behavior Details)")
print("\nHere are the first 6 numbers of the final behavior footprint:")
print(agent_ready_features[0][:6])

--- 1. UPGRADING THE LSTM ROBOT (ADJUSTING THE PLUG) ---
✅ LSTM Robot Upgraded successfully!

--- 2. FEEDING DATA TO THE LSTM BRAIN ---
Feeding data with shape: (1, 144, 512) into the LSTM...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 382ms/step

🎉 BINGO! FINAL FEATURES EXTRACTED! 🎉
📊 AGENT-READY SHAPE: (1, 512)
Format: (Number of Videos, Extracted Behavior Details)

Here are the first 6 numbers of the final behavior footprint:
[0.         0.         0.5474293  0.28662366 0.         0.        ]


In [9]:
import os

print("--- SAVING THE AI MODEL FOR BACKEND DEPLOYMENT ---")

# Tentukan lokasi penyimpanan di Google Drive Anda agar tidak hilang saat Colab ditutup
save_path = '/content/drive/MyDrive/Autism_Agent_Project/autism_behavior_extractor.keras'

# Simpan modelnya
final_headless_model.save(save_path)

# Verifikasi apakah file berhasil dibuat
if os.path.exists(save_path):
    print(f"✅  Model berhasil disimpan di:")
    print(f"   {save_path}")
    print("\nFile .keras ini adalah 'Otak' yang akan kita bawa ke Google Cloud Run!")
else:
    print("❌ Gagal menyimpan model. Cek kembali path Google Drive Anda.")

--- SAVING THE AI MODEL FOR BACKEND DEPLOYMENT ---
✅  Model berhasil disimpan di:
   /content/drive/MyDrive/Autism_Agent_Project/autism_behavior_extractor.keras

File .keras ini adalah 'Otak' yang akan kita bawa ke Google Cloud Run!


In [10]:
import os
import librosa
import numpy as np

print("--- 1. PREPARING AUDIO DATASET (CHUNKING & MFCC) ---")

# Define paths based on your Google Drive structure
base_audio_path = '/content/drive/MyDrive/Autism_Agent_Project/data_audio_noise'
asd_audio_path = os.path.join(base_audio_path, 'ASD_Audio') # This will target record1.mp3 & record2.mp3
urban_audio_path = os.path.join(base_audio_path, 'UrbanSound8K/audio/fold1') # Make sure this folder exists!

def extract_audio_chunks(folder_path, label, chunk_duration=3.0, sr=16000):
    features = []
    labels = []

    if not os.path.exists(folder_path):
        print(f"⚠️ Folder not found: {folder_path}")
        return np.array(features), np.array(labels)

    for file_name in os.listdir(folder_path):
        # Explicitly checking for .mp3 and .wav files
        if file_name.endswith('.wav') or file_name.endswith('.mp3'):
            file_path = os.path.join(folder_path, file_name)
            try:
                # Load audio file (librosa automatically handles mp3 decoding in Colab)
                y, _ = librosa.load(file_path, sr=sr)

                # Calculate how many 3-second chunks we can extract from the file
                samples_per_chunk = int(sr * chunk_duration)
                num_chunks = len(y) // samples_per_chunk

                for i in range(num_chunks):
                    # Slice the audio into chunks
                    start_sample = i * samples_per_chunk
                    end_sample = start_sample + samples_per_chunk
                    y_chunk = y[start_sample:end_sample]

                    # Extract MFCC (Mel-frequency cepstral coefficients) - Standard for Audio AI
                    mfcc = librosa.feature.mfcc(y=y_chunk, sr=sr, n_mfcc=40)
                    mfcc_scaled = np.mean(mfcc.T, axis=0) # Average the features over time

                    features.append(mfcc_scaled)
                    labels.append(label)
            except Exception as e:
                print(f"❌ Error processing {file_name}: {e}")

    return np.array(features), np.array(labels)

print("Processing ASD audio files (Label 1)... this might take a moment.")
asd_features, asd_labels = extract_audio_chunks(asd_audio_path, label=1)
print(f"✅ Successfully created {len(asd_features)} samples from ASD files.")

print("Processing UrbanSound/Noise files (Label 0)...")
# To prevent class imbalance, we limit the noise samples to match the number of ASD samples
noise_features, noise_labels = extract_audio_chunks(urban_audio_path, label=0)
if len(noise_features) > len(asd_features):
    noise_features = noise_features[:len(asd_features)]
    noise_labels = noise_labels[:len(asd_labels)]
print(f"✅ Successfully created {len(noise_features)} Noise samples.")

# Concatenate ASD and Noise data
if len(asd_features) > 0 and len(noise_features) > 0:
    X_audio = np.concatenate((asd_features, noise_features), axis=0)
    # Reshape for Conv1D input: (number_of_samples, number_of_features, 1_channel)
    X_audio = np.expand_dims(X_audio, axis=2)
    print(f"📊 FINAL AUDIO DATA SHAPE: {X_audio.shape}")
else:
    print("⚠️ WARNING: Audio data is empty. Please check your Drive folders.")

--- 1. PREPARING AUDIO DATASET (CHUNKING & MFCC) ---
Processing ASD audio files (Label 1)... this might take a moment.
✅ Successfully created 106 samples from ASD files.
Processing UrbanSound/Noise files (Label 0)...
✅ Successfully created 106 Noise samples.
📊 FINAL AUDIO DATA SHAPE: (212, 40, 1)


In [12]:
from keras.models import Model
from keras.layers import Dense, Conv1D, Dropout, Input, Flatten
import numpy as np
import os

print("--- 2. BUILDING & TRAINING THE 'EARS' OF AI ---")

# Combine the labels we created in Cell 1
y_audio = np.concatenate((asd_labels, noise_labels), axis=0)

# Build the Neural Network architecture
audio_input = Input(shape=(40, 1), name='audio_input_mfcc')

x = Conv1D(64, kernel_size=5, padding='same', activation='relu')(audio_input)
x = Conv1D(128, kernel_size=5, padding='same', activation='relu')(x)
x = Dropout(0.3)(x)
x = Flatten()(x)

# Feature Extractor Layer
audio_features = Dense(256, activation='relu', name='audio_behavior_features')(x)

# Classifier Layer (Temporary mouth for training)
# PERBAIKAN: Menggunakan 'softmax' sebagai fungsi aktivasi
audio_classifier = Dense(2, activation='softmax', name='audio_classifier')(audio_features)

# Create the full model for training purposes
training_model = Model(inputs=audio_input, outputs=audio_classifier)

# Loss function 'sparse_categorical_crossentropy' diletakkan di compile, bukan di layer
training_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("\n🚀 Training the AI Ears on your 212 audio samples...")
# We train for 15 epochs. Since the data is small, this will be very fast!
training_model.fit(X_audio, y_audio, epochs=15, batch_size=16, validation_split=0.2)

print("\n--- 3. EXTRACTING HEADLESS MODEL & SAVING TO DRIVE ---")
# The Headless Model (We cut off the classifier layer, keeping only the feature extractor)
headless_audio_model = Model(inputs=audio_input, outputs=audio_features, name="Headless_Audio_Extractor")

print("\nHeadless Audio Model Architecture:")
headless_audio_model.summary()

# Define the save path in Google Drive
audio_save_path = '/content/drive/MyDrive/Autism_Agent_Project/autism_audio_extractor.keras'
headless_audio_model.save(audio_save_path)

# Verify if the model was saved successfully
if os.path.exists(audio_save_path):
    print(f"\n✅ Trained Audio Model successfully saved at:\n   {audio_save_path}")
    print("\n🎉 CONGRATULATIONS! You now have custom-trained 'Eyes' and 'Ears' ready for Cloud Run!")
else:
    print("\n❌ Failed to save the model. Please check your Google Drive path.")

--- 2. BUILDING & TRAINING THE 'EARS' OF AI ---

🚀 Training the AI Ears on your 212 audio samples...
Epoch 1/15
11/11 ━━━━━━━━━━━━━━━━━━━━ 6s 247ms/step - accuracy: 0.5675 - loss: 5.3620 - val_accuracy: 0.8140 - val_loss: 0.3015
Epoch 2/15
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9356 - loss: 0.2341 - val_accuracy: 0.9302 - val_loss: 0.1805
Epoch 3/15
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9450 - loss: 0.1554 - val_accuracy: 0.9535 - val_loss: 0.1127
Epoch 4/15
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9679 - loss: 0.0704 - val_accuracy: 0.9535 - val_loss: 0.1750
Epoch 5/15
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9963 - loss: 0.0356 - val_accuracy: 0.9535 - val_loss: 0.1456
Epoch 6/15
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9917 - loss: 0.0234 - val_accuracy: 0.9302 - val_loss: 0.2293
Epoch 7/15
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9957 - loss: 0.0189 - val_accuracy: 0.9302 - val_loss: 0.2956
Epoch 8/15
11/

Model: "Headless_Audio_Extractor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ audio_input_mfcc (InputLayer)   │ (None, 40, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 40, 64)         │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 40, 128)        │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 40, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 5120)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ audio_behavior_features (Dense) │ (None, 256)            │     1,310,976 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,352,448 (5.16 MB)

 Trainable params: 1,352,448 (5.16 MB)

 Non-trainable params: 0 (0.00 B)


✅ Trained Audio Model successfully saved at:
   /content/drive/MyDrive/Autism_Agent_Project/autism_audio_extractor.keras

🎉 CONGRATULATIONS! You now have custom-trained 'Eyes' and 'Ears' ready for Cloud Run!
